In [2]:
import re
import json
import unicodedata
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

SCOPUS_FILE = "Burnout (Scopus).csv"
WOS_FILE = "Burnout (WOS).bib"


def parse_bibtex_simple(path):
    text = Path(path).read_text(encoding="utf-8-sig", errors="replace")
    entries = []
    pos = 0
    while True:
        m = re.search(r"@([A-Za-z]+)\s*\{", text[pos:])
        if not m:
            break
        typ = m.group(1)
        brace_start = pos + m.end() - 1
        depth = 0
        end = None
        for i in range(brace_start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        if end is None:
            break
        entries.append((typ, text[brace_start + 1:end]))
        pos = end + 1

    def parse_fields(typ, body):
        parts = body.split(",", 1)
        key = parts[0].strip()
        rest = parts[1] if len(parts) > 1 else ""
        fields = {"bibtex_key": key, "bibtex_type": typ}
        i, n = 0, len(rest)
        while i < n:
            while i < n and rest[i] in " \t\r\n,":
                i += 1
            if i >= n:
                break
            m = re.match(r"([A-Za-z0-9_\-]+)\s*=\s*", rest[i:])
            if not m:
                i += 1
                continue
            fname = m.group(1).lower()
            i += m.end()
            if i < n and rest[i] == "{":
                i += 1
                depth = 1
                start = i
                while i < n and depth > 0:
                    if rest[i] == "{":
                        depth += 1
                    elif rest[i] == "}":
                        depth -= 1
                    i += 1
                val = rest[start:i - 1]
            elif i < n and rest[i] == '"':
                i += 1
                start = i
                while i < n and rest[i] != '"':
                    i += 1
                val = rest[start:i]
                i += 1
            else:
                start = i
                while i < n and rest[i] != ",":
                    i += 1
                val = rest[start:i]
            fields[fname] = " ".join(str(val).replace("\\%", "%").split())
        return fields

    return pd.DataFrame([parse_fields(t, b) for t, b in entries])


def norm_doi(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = re.sub(r"^https?://(dx\.)?doi\.org/", "", s)
    return s.strip(" .;")


def norm_title(x):
    if pd.isna(x):
        return ""
    s = str(x).lower()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


def add_box(ax, x, y, w, h, text, color, fontsize=9.3):
    ax.add_patch(Rectangle((x, y), w, h, facecolor=color, edgecolor="none"))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True)


def arrow(ax, x1, y1, x2, y2):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle="-|>", lw=1.2, color="black", mutation_scale=14, shrinkA=0, shrinkB=0),
    )


def draw_prisma(summary, reports, excluded, filename_base, subtitle):
    fig, ax = plt.subplots(figsize=(8, 12))
    fig.patch.set_facecolor("#e8e8e8")
    ax.set_facecolor("#e8e8e8")
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 18)
    ax.axis("off")

    azul = "#b9dde8"
    rojo = "#ec8f8f"
    verde = "#92e68f"
    cx, cw, ch = 1.45, 4.75, 1.12
    rx, rw = 6.65, 2.85
    y1, y2, y3, y4, y5, y6 = 15.25, 13.35, 11.45, 9.55, 7.65, 4.85

    ax.text(5, 17.55, "Diagrama de flujo PRISMA 2020", ha="center", va="center", fontsize=15, fontweight="bold")
    ax.text(5, 17.15, subtitle, ha="center", va="center", fontsize=9.5)

    add_box(ax, cx, y1, cw, 1.42,
            f"Registros identificados mediante búsqueda\nen bases de datos\n(n = {summary['total_identified']})\nWoS (n = {summary['wos']})\nScopus (n = {summary['scopus']})", azul, 9.1)
    add_box(ax, rx, y1 + 0.1, rw, 1.2,
            f"Duplicados eliminados\n(n = {summary['duplicates_removed']})", rojo, 8.8)
    add_box(ax, cx, y2, cw, ch, f"Registros tras eliminar duplicados\n(n = {summary['unique_records']})", azul, 9.8)
    add_box(ax, cx, y3, cw, ch, f"Registros cribados (Screening)\n(n = {summary['unique_records']})", azul, 9.8)
    add_box(ax, rx, y3, rw, ch, f"Registros excluidos\n(n = {excluded})", rojo, 9.8)
    add_box(ax, cx, y4, cw, ch, f"Informes buscados para recuperación\n(n = 14)", azul, 9.8)
    add_box(ax, rx, y4, rw, ch, "Informes no recuperados\n(n = 2)", rojo, 9.2)
    add_box(ax, cx, y5, cw, ch, "Informes evaluados para elegibilidad\n(n = 12)", azul, 9.2)
    add_box(ax, rx, y5 - 0.1, rw, 1.32, "Informes excluidos (Irrelevantes\npara la investigación)\n(n = 6)", rojo, 9.2)
    add_box(ax, cx + 0.2, y6, cw - 0.4, 1.3, "Estudios incluidos en la revisión\n(n = 6)", verde, 9.8)

    center = cx + cw / 2
    arrow(ax, cx + cw, y1 + 0.71, rx, y1 + 0.71)
    arrow(ax, center, y1, center, y2 + ch)
    arrow(ax, center, y2, center, y3 + ch)
    arrow(ax, cx + cw, y3 + ch / 2, rx, y3 + ch / 2)
    arrow(ax, center, y3, center, y4 + ch)
    arrow(ax, cx + cw, y4 + ch / 2, rx, y4 + ch / 2)
    arrow(ax, center, y4, center, y5 + ch)
    arrow(ax, cx + cw, y5 + ch / 2, rx, y5 + 0.56)
    arrow(ax, center, y5, center, y6 + 1.3)


    for ext in ["png", "jpg", "svg"]:
        plt.savefig(f"{filename_base}.{ext}", dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

def main():
    scopus_raw = pd.read_csv(SCOPUS_FILE)
    wos_raw = parse_bibtex_simple(WOS_FILE)

    scopus = pd.DataFrame({
        "source": "Scopus",
        "record_id": scopus_raw.get("EID", ""),
        "authors": scopus_raw.get("Authors", ""),
        "title": scopus_raw.get("Title", ""),
        "year": scopus_raw.get("Year", ""),
        "journal": scopus_raw.get("Source title", ""),
        "doi": scopus_raw.get("DOI", ""),
        "link": scopus_raw.get("Link", ""),
        "abstract": scopus_raw.get("Abstract", ""),
        "doc_type": scopus_raw.get("Document Type", ""),
    })
    wos = pd.DataFrame({
        "source": "Web of Science",
        "record_id": wos_raw.get("unique-id", ""),
        "authors": wos_raw.get("author", ""),
        "title": wos_raw.get("title", ""),
        "year": wos_raw.get("year", ""),
        "journal": wos_raw.get("journal", ""),
        "doi": wos_raw.get("doi", ""),
        "link": wos_raw.get("url", ""),
        "abstract": wos_raw.get("abstract", ""),
        "doc_type": wos_raw.get("bibtex_type", ""),
    })

    combined = pd.concat([scopus, wos], ignore_index=True)
    for c in ["title", "authors", "journal", "doi", "link", "abstract", "doc_type", "record_id"]:
        combined[c] = combined[c].fillna("").astype(str)
    combined["year"] = combined["year"].fillna("").astype(str)
    combined["doi_norm"] = combined["doi"].apply(norm_doi)
    combined["title_norm"] = combined["title"].apply(norm_title)
    combined["duplicate_removed"] = (
        (combined["doi_norm"].ne("") & combined.duplicated("doi_norm", keep="first")) |
        (combined["title_norm"].ne("") & combined.duplicated("title_norm", keep="first"))
    )
    unique = combined[~combined["duplicate_removed"]].copy().reset_index(drop=True)
    unique["text"] = (unique["title"] + " " + unique["abstract"] + " " + unique["journal"] + " " + unique["doc_type"]).str.lower()

    def has(pat):
        return unique["text"].str.contains(pat, case=False, regex=True, na=False)
    def has_title(pat):
        return unique["title"].str.lower().str.contains(pat, case=False, regex=True, na=False)

    pop_pat = r"\b(university|college|undergraduate|higher education|medical student|nursing student|dental student|pharmacy student|student|students|school|academic|educational)\b|estudiante|universitar|educaci[oó]n superior"
    mental_pat = r"\b(burnout|academic stress|stress|mental health|anxiety|depression|wellbeing|well-being|emotional exhaustion|psychological|fatigue|distress)\b|agotamiento|estres|estr[eé]s|salud mental|ansiedad|depresi[oó]n|bienestar"
    data_pat = r"\b(machine learning|deep learning|artificial intelligence|predictive|prediction|predictor|predictors|classification|classifier|data mining|data science|random forest|xgboost|support vector|svm|neural network|regression|algorithm|model|models|analytics|knn|decision tree|naive bayes|bayesian|clustering|k-means|data analysis|network analysis)\b|aprendizaje autom[aá]tico|inteligencia artificial|modelo predictivo|clasificaci[oó]n|regresi[oó]n"
    burnout_pat = r"\b(burnout|academic burnout|student burnout|school burnout|emotional exhaustion|maslach|mbi|burnout inventory)\b|agotamiento"
    student_context_pat = r"\b(university student|university students|college student|college students|undergraduate|higher education students|students in higher education|medical student|medical students|nursing student|nursing students|dental student|dental students|pharmacy student|pharmacy students|surgical instrumentation students|student|students)\b|estudiantes universitarios|universitar"
    model_pat = r"\b(machine learning|deep learning|artificial intelligence|predict|prediction|predictive|predictor|predictors|model|models|regression|classification|classifier|random forest|xgboost|svm|support vector|neural network|data mining|data analysis|network analysis|clustering|k-means|factors associated|associated factors|association|associations|relationship|relationships|influence|role of)\b|modelo|predic|regresi|clasificaci"
    review_doc_pat = r"\b(systematic review|literature review|bibliometric|protocol|editorial|letter|conference paper|book chapter|proceeding)\b|revisi[oó]n|bibliom[eé]tric"
    nonstudent_title_pat = r"\b(teacher|teachers|academic staff|academics|clinician|clinicians|faculty|professor|professors|workers|employees|social workers|primary school teachers|university teachers)\b"

    unique["has_population_edu"] = has(pop_pat)
    unique["has_mental_health_or_burnout"] = has(mental_pat)
    unique["has_data_science_or_prediction"] = has(data_pat)
    unique["has_burnout_specific"] = has(burnout_pat)
    unique["has_student_context"] = has(student_context_pat)
    unique["has_model_or_predictor_context"] = has(model_pat)
    unique["is_review_or_non_article_like"] = has(review_doc_pat)
    unique["has_nonstudent_population_in_title"] = has_title(nonstudent_title_pat)
    unique["is_article"] = unique["doc_type"].str.lower().eq("article")

    unique["screen_broad"] = unique["has_population_edu"] & unique["has_mental_health_or_burnout"] & unique["has_data_science_or_prediction"]
    unique["screen_recommended"] = (
        unique["has_burnout_specific"] & unique["has_student_context"] & unique["has_model_or_predictor_context"] &
        unique["is_article"] & ~unique["is_review_or_non_article_like"] & ~unique["has_nonstudent_population_in_title"]
    )

    unique["priority_auto"] = "No candidato"
    unique.loc[unique["screen_broad"], "priority_auto"] = "Baja"
    unique.loc[unique["screen_broad"] & unique["has_burnout_specific"], "priority_auto"] = "Media"
    unique.loc[unique["screen_recommended"], "priority_auto"] = "Alta"
    unique["decision_manual"] = ""
    unique["reason_exclusion"] = ""
    unique["doi_link"] = unique["doi_norm"].apply(lambda d: f"https://doi.org/{d}" if d else "")
    unique["google_scholar_search"] = unique["title"].apply(lambda t: "https://scholar.google.com/scholar?q=" + re.sub(r"\s+", "+", t.strip()) if t.strip() else "")

    priority_order = {"Alta": 0, "Media": 1, "Baja": 2, "No candidato": 3}
    unique["priority_order"] = unique["priority_auto"].map(priority_order).fillna(9)

    broad = unique[unique["screen_broad"]].sort_values(["priority_order", "year", "title"], ascending=[True, False, True])
    recommended = unique[unique["screen_recommended"]].sort_values(["priority_order", "year", "title"], ascending=[True, False, True])
    cols = ["decision_manual", "reason_exclusion", "priority_auto", "source", "year", "title", "authors", "journal", "doc_type", "doi", "doi_link", "google_scholar_search", "abstract"]

    combined.to_csv("combined_all_records_v2.csv", index=False, encoding="utf-8-sig")
    unique.to_csv("combined_unique_records_screened_v2.csv", index=False, encoding="utf-8-sig")
    broad[cols].to_csv("informes_candidatos_PRISMA_amplio_v2.csv", index=False, encoding="utf-8-sig")
    recommended[cols].to_csv("informes_candidatos_PRISMA_recomendado_v2.csv", index=False, encoding="utf-8-sig")

    summary = {
        "scopus": int((combined["source"] == "Scopus").sum()),
        "wos": int((combined["source"] == "Web of Science").sum()),
        "total_identified": int(len(combined)),
        "duplicates_removed": int(combined["duplicate_removed"].sum()),
        "unique_records": int(len(unique)),
        "broad_candidates": int(len(broad)),
        "broad_excluded": int(len(unique) - len(broad)),
        "recommended_candidates": int(len(recommended)),
        "recommended_excluded": int(len(unique) - len(recommended)),
        "abstracts_scopus": int(((combined["source"] == "Scopus") & combined["abstract"].str.strip().ne("")).sum()),
        "abstracts_wos": int(((combined["source"] == "Web of Science") & combined["abstract"].str.strip().ne("")).sum()),
    }
    Path("prisma_summary_v2.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    draw_prisma(summary, summary["broad_candidates"], summary["broad_excluded"], "prisma_burnout_v2_amplio", "Filtro amplio: candidatos para revisión preliminar")
    draw_prisma(summary, summary["recommended_candidates"], summary["recommended_excluded"], "prisma_burnout_v2_recomendado", "Filtro recomendado: burnout académico + estudiantes + modelado/predictores")

    print(json.dumps(summary, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()

/tmp/ipykernel_26302/4087424700.py:205: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return unique["text"].str.contains(pat, case=False, regex=True, na=False)
/tmp/ipykernel_26302/4087424700.py:205: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return unique["text"].str.contains(pat, case=False, regex=True, na=False)
/tmp/ipykernel_26302/4087424700.py:205: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return unique["text"].str.contains(pat, case=False, regex=True, na=False)
/tmp/ipykernel_26302/4087424700.py:205: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return unique["text"].str.contains(pat, case=False, regex=True, na=False)
/tmp

{
  "scopus": 134,
  "wos": 886,
  "total_identified": 1020,
  "duplicates_removed": 2,
  "unique_records": 1018,
  "broad_candidates": 122,
  "broad_excluded": 896,
  "recommended_candidates": 14,
  "recommended_excluded": 1004,
  "abstracts_scopus": 134,
  "abstracts_wos": 883
}
